# Find Your Top 3 Cost Drivers

> **No support provided.** This code is for reference only. Review, test, and modify before any production use.

**Pair-programmed by:** SE Community + Cortex Code

---

## What This Workbook Does

Before you resize your warehouse, run these queries. Most "slow query" problems are actually **pruning problems** -- Snowflake is scanning partitions it doesn't need.

This workbook helps you:
1. **Find** your top cost drivers (queries that scan the most data)
2. **Diagnose** the root cause (missing clustering, no search optimization, anti-patterns)
3. **Fix** without resizing (clustering keys, SOS, query rewrites)

**Time:** ~30 minutes | **Result:** Faster queries, lower costs, no warehouse resize

## Prerequisites

You need access to `SNOWFLAKE.ACCOUNT_USAGE` views. This typically means:
- `ACCOUNTADMIN` role, OR
- A custom role with `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database

```sql
-- Grant access to a custom role (run as ACCOUNTADMIN)
GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE MY_ANALYST_ROLE;
```

---
# Section 1: Find Your Top Cost Drivers

These queries identify which queries are consuming the most resources. The key metric is **partition scan ratio** -- if you're scanning 100% of partitions, you're doing full table scans.

> **Privacy note:** `query_text` previews may contain literal values (PII, tokens, connection strings). Treat output from these diagnostic queries as sensitive -- scrub before sharing screenshots or exports.

In [ ]:
-- Top 10 queries by execution time (last 30 days)
-- Look for queries with high pct_scanned -- these are your optimization targets
SELECT 
    query_id,
    SUBSTR(query_text, 1, 100) AS query_preview,
    ROUND(total_elapsed_time / 1000, 2) AS seconds,
    partitions_scanned,
    partitions_total,
    ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
    ROUND(bytes_scanned / POWER(1024, 3), 2) AS gb_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time > DATEADD(day, -30, CURRENT_TIMESTAMP())
  AND query_type = 'SELECT'
  AND partitions_total > 0
ORDER BY total_elapsed_time DESC
LIMIT 10

In [ ]:
-- Top 10 queries by bytes spilled (memory pressure)
-- Spillage means the query ran out of memory and had to use disk -- very slow
SELECT 
    query_id,
    SUBSTR(query_text, 1, 100) AS query_preview,
    ROUND(total_elapsed_time / 1000, 2) AS seconds,
    ROUND(bytes_spilled_to_local_storage / POWER(1024, 3), 2) AS gb_spilled_local,
    ROUND(bytes_spilled_to_remote_storage / POWER(1024, 3), 2) AS gb_spilled_remote,
    warehouse_size
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time > DATEADD(day, -30, CURRENT_TIMESTAMP())
  AND query_type = 'SELECT'
  AND (bytes_spilled_to_local_storage > 0 OR bytes_spilled_to_remote_storage > 0)
ORDER BY (bytes_spilled_to_local_storage + bytes_spilled_to_remote_storage) DESC
LIMIT 10

In [ ]:
-- Top 10 queries by queue time (warehouse contention)
-- High queue time means queries are waiting for compute
SELECT 
    query_id,
    SUBSTR(query_text, 1, 100) AS query_preview,
    ROUND(queued_provisioning_time / 1000, 2) AS queue_provisioning_sec,
    ROUND(queued_overload_time / 1000, 2) AS queue_overload_sec,
    ROUND(total_elapsed_time / 1000, 2) AS total_seconds,
    warehouse_name,
    warehouse_size
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time > DATEADD(day, -30, CURRENT_TIMESTAMP())
  AND (queued_provisioning_time > 1000 OR queued_overload_time > 1000)
ORDER BY (queued_provisioning_time + queued_overload_time) DESC
LIMIT 10

---
# Section 2: Diagnose the Root Cause

## What These Numbers Mean

| Symptom | Diagnosis | Fix |
|---------|-----------|-----|
| `pct_scanned` > 80% | No partition pruning | Add `CLUSTER BY` on filter columns |
| `gb_spilled` > 0 | Query exceeds memory | Rewrite query OR cluster to reduce scan |
| `queue_time` high | Warehouse contention | Optimize queries first, THEN consider multi-cluster |
| Point lookups slow | No search optimization | Add `SEARCH OPTIMIZATION` on equality columns |

In [ ]:
-- Tables with poor partition pruning (clustering candidates)
-- avg_scan_ratio close to 1.0 means nearly full table scans every time
SELECT 
    database_name,
    COUNT(*) AS query_count,
    ROUND(AVG(partitions_scanned / NULLIF(partitions_total, 0)), 2) AS avg_scan_ratio,
    SUM(partitions_scanned) AS total_partitions_scanned,
    ROUND(SUM(bytes_scanned) / POWER(1024, 4), 2) AS tb_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time > DATEADD(day, -30, CURRENT_TIMESTAMP())
  AND query_type = 'SELECT'
  AND partitions_total > 100
GROUP BY database_name
HAVING avg_scan_ratio > 0.5
ORDER BY total_partitions_scanned DESC
LIMIT 20

In [ ]:
-- Large tables without clustering keys (low-hanging fruit)
-- These are your prime candidates for CLUSTER BY
SELECT 
    table_catalog || '.' || table_schema || '.' || table_name AS full_name,
    row_count,
    ROUND(bytes / POWER(1024, 3), 2) AS size_gb,
    clustering_key
FROM SNOWFLAKE.ACCOUNT_USAGE.TABLES
WHERE deleted IS NULL
  AND row_count > 1000000
  AND (clustering_key IS NULL OR clustering_key = '')
  AND table_type = 'BASE TABLE'
ORDER BY bytes DESC
LIMIT 20

In [ ]:
-- Tables with Search Optimization enabled vs disabled
-- Large tables with frequent point lookups benefit from SOS
SELECT 
    table_catalog || '.' || table_schema || '.' || table_name AS full_name,
    row_count,
    ROUND(bytes / POWER(1024, 3), 2) AS size_gb,
    search_optimization,
    ROUND(search_optimization_bytes / POWER(1024, 3), 2) AS sos_size_gb
FROM SNOWFLAKE.ACCOUNT_USAGE.TABLES
WHERE deleted IS NULL
  AND row_count > 10000000
  AND table_type = 'BASE TABLE'
ORDER BY bytes DESC
LIMIT 20

---
# Section 3: Fix Without Resizing

## Decision Tree

```
Query is slow
    │
    ├── Check pct_scanned
    │   ├── > 80% → Add CLUSTER BY on filter columns
    │   └── < 20% → Pruning is working, look elsewhere
    │
    ├── Check spillage
    │   ├── > 0 GB → Rewrite query (reduce columns, add filters)
    │   └── 0 GB → Memory is fine
    │
    └── Point lookups slow?
        ├── Yes → Add SEARCH OPTIMIZATION on equality columns
        └── No → Check query profile for join explosions
```

### Fix 1: Add Clustering Key

Clustering organizes data physically by the columns you filter on most. Choose columns that:
- Appear in `WHERE` clauses frequently
- Have high cardinality (dates, IDs) but aren't unique per row
- Are used together in filters

**Column order matters:** Put the most selective column first.

In [ ]:
-- Template: Add clustering key
-- Uncomment and replace MY_DATABASE.MY_SCHEMA.MY_TABLE and columns

-- ALTER TABLE MY_DATABASE.MY_SCHEMA.MY_TABLE 
--   CLUSTER BY (date_column, region_column);

-- Check clustering status
-- SHOW TABLES LIKE 'MY_TABLE' IN SCHEMA MY_DATABASE.MY_SCHEMA;

SELECT 'Uncomment the ALTER TABLE statement above and customize for your table' AS instruction

### Fix 2: Enable Search Optimization

Search Optimization Service (SOS) accelerates point lookups -- queries like `WHERE customer_id = '12345'`. It's Snowflake's answer to traditional indexes.

**Best for:**
- Equality predicates (`=`, `IN`)
- VARIANT/OBJECT path lookups
- Geospatial queries
- Substring searches (`LIKE '%pattern%'`)

In [ ]:
-- Template: Enable Search Optimization
-- Uncomment and replace MY_DATABASE.MY_SCHEMA.MY_TABLE and columns

-- Full table SOS (all columns)
-- ALTER TABLE MY_DATABASE.MY_SCHEMA.MY_TABLE ADD SEARCH OPTIMIZATION;

-- Targeted SOS (specific columns -- more cost-effective)
-- ALTER TABLE MY_DATABASE.MY_SCHEMA.MY_TABLE 
--   ADD SEARCH OPTIMIZATION ON EQUALITY(customer_id, order_id);

-- Check SOS status
-- DESCRIBE SEARCH OPTIMIZATION ON MY_DATABASE.MY_SCHEMA.MY_TABLE;

SELECT 'Uncomment the ALTER TABLE statement above and customize for your table' AS instruction

### Fix 3: Query Anti-Patterns

| Anti-Pattern | Problem | Fix |
|--------------|---------|-----|
| `SELECT *` | Scans all columns | Select only needed columns |
| `WHERE DATE(timestamp_col) = '2024-01-01'` | Function on column breaks pruning | `WHERE timestamp_col >= '2024-01-01' AND timestamp_col < '2024-01-02'` |
| No date filter on time-series table | Scans entire history | Add explicit date range |
| `LIKE '%pattern%'` without SOS | Full scan | Enable SOS with SUBSTRING method |
| Large table JOINs without filters | Cartesian explosion | Filter before joining |

---
# Section 4: Verify Improvement

After applying fixes, re-run your problem query and compare the metrics.

In [ ]:
-- Compare before/after for a specific query
-- Replace the query IDs with your actual before/after IDs

SELECT 
    query_id,
    start_time,
    ROUND(total_elapsed_time / 1000, 2) AS seconds,
    partitions_scanned,
    partitions_total,
    ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
    ROUND(bytes_scanned / POWER(1024, 3), 2) AS gb_scanned,
    ROUND(bytes_spilled_to_local_storage / POWER(1024, 3), 2) AS gb_spilled
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE query_id IN (
    '01234567-89ab-cdef-0123-456789abcdef',  -- BEFORE
    'fedcba98-7654-3210-fedc-ba9876543210'   -- AFTER
)
ORDER BY start_time

In [ ]:
-- Check clustering depth for a table (lower is better)
-- Uncomment and replace with your table name

-- SELECT SYSTEM$CLUSTERING_INFORMATION('MY_DATABASE.MY_SCHEMA.MY_TABLE');

SELECT 'Uncomment the SYSTEM$CLUSTERING_INFORMATION call above' AS instruction

---
# Section 5: Ongoing Monitoring

Set up a weekly check to catch regressions before they become problems.

In [ ]:
-- Weekly cost driver summary
-- Schedule this as a Task to run every Monday
SELECT 
    DATE_TRUNC('week', start_time) AS week,
    COUNT(*) AS query_count,
    ROUND(AVG(partitions_scanned / NULLIF(partitions_total, 0)), 2) AS avg_scan_ratio,
    ROUND(SUM(bytes_scanned) / POWER(1024, 4), 2) AS tb_scanned,
    ROUND(SUM(bytes_spilled_to_local_storage + bytes_spilled_to_remote_storage) / POWER(1024, 4), 2) AS tb_spilled
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time > DATEADD(week, -8, CURRENT_TIMESTAMP())
  AND query_type = 'SELECT'
GROUP BY week
ORDER BY week DESC

---
# References

### Snowflake Documentation

- [Clustering Keys & Clustered Tables](https://docs.snowflake.com/en/user-guide/tables-clustering-keys)
- [Search Optimization Service](https://docs.snowflake.com/en/user-guide/search-optimization-service)
- [Query Profile](https://docs.snowflake.com/en/user-guide/ui-query-profile)
- [QUERY_HISTORY View](https://docs.snowflake.com/en/sql-reference/account-usage/query_history)
- [SYSTEM$CLUSTERING_INFORMATION](https://docs.snowflake.com/en/sql-reference/functions/system_clustering_information)